# 24 · ANN Indexing Internals

**Idea:** real vector databases don't scan every vector — they use Approximate Nearest Neighbor (ANN) indexes that trade a little recall for a lot of speed. Two dominant ideas, built here as tiny NumPy implementations:

- **IVF** (inverted file): cluster vectors, then search only the nearest few clusters.
- **Scalar quantization**: store each dimension as one byte instead of a float32 — 4× less memory, small accuracy hit.

`recall_at_k` measures exactly what the approximation costs versus exact brute force.

> Pure NumPy over Ollama embeddings — no cloud keys. One optional cell uses `hnswlib` if installed.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Embed the corpus into a normalized matrix and pick a query vector. This matrix is what a vector index is built over.

In [ ]:
import numpy as np
from rag_lab import load_markdown, embed_texts

chunks = load_markdown()
matrix = np.array(embed_texts([c.text for c in chunks], provider='ollama'))
query = embed_texts(['how does hybrid search combine keyword and vector retrieval?'], provider='ollama')[0]
print('matrix shape:', matrix.shape)

Establish ground truth with exact **brute force**, then build an **IVF** index and search only the nearest clusters. `recall_at_k` tells us how much recall the speedup cost.

In [ ]:
from rag_lab import brute_force, IVFIndex, recall_at_k

exact = brute_force(matrix, query, k=5)
ivf = IVFIndex(nlist=8, nprobe=2).build(matrix)
approx = ivf.search(query, k=5)

print('exact top-5 :', exact)
print('IVF   top-5 :', approx)
print('recall@5    :', recall_at_k(approx, exact))

More probes → higher recall, less speedup. This `nprobe` dial is the recall/latency trade every IVF index exposes.

In [ ]:
for nprobe in (1, 2, 4, 8):
    idx = IVFIndex(nlist=8, nprobe=nprobe).build(matrix)
    r = recall_at_k(idx.search(query, k=5), exact)
    print(f'nprobe={nprobe} -> recall@5 = {r:.2f}')

**Quantization:** compress float32 vectors to uint8 codes. Memory drops ~4×; measure the recall it costs after reconstructing.

In [ ]:
from rag_lab import quantize, dequantize

codes, lo, span = quantize(matrix)
recon = dequantize(codes, lo, span)
q_approx = brute_force(recon, query, k=5)

print(f'float32 bytes: {matrix.nbytes:>7}  ->  uint8 bytes: {codes.nbytes:>7}  ({matrix.nbytes/codes.nbytes:.1f}x smaller)')
print('recall@5 after quantization:', recall_at_k(q_approx, exact))

Optionally compare against a real graph index (HNSW) via `hnswlib`. Skips if not installed.

In [ ]:
try:
    from rag_lab import hnsw_search
    h = hnsw_search(matrix, query, k=5)
    print('HNSW top-5:', h, '| recall@5:', recall_at_k(h, exact))
except Exception as e:
    print('hnswlib skipped:', type(e).__name__, e)

**Takeaway:** "vector search" at scale is a recall/speed/memory triangle. IVF trades recall for speed via `nprobe`; quantization trades recall for memory. Knowing these internals lets you tune a production index (Chroma/FAISS/pgvector) instead of treating it as magic.

Next up → `25_corrective_rag.ipynb`, where the pipeline grades its own retrieval and retries.